# Deepfake Dataset Downloader — Google Colab

Downloads **FaceForensics++** and **Celeb-DF** directly to your Google Drive.

| Dataset | Source | Size | Approval? |
|---------|--------|------|-----------|
| **FaceForensics++** | Kaggle | ~2.9 GB | No (free account) |
| **Celeb-DF v2** | Google Drive | ~10 GB | No (direct link) |

Run all cells top to bottom. Everything stays on your Drive — nothing fills up the VM disk.

In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/deepfake_datasets"
!mkdir -p "$DRIVE_BASE"
print(f"Datasets will be saved to: {DRIVE_BASE}")

In [ ]:
#@title 2. Install Packages & Import Helpers
!pip install -q gdown kaggle

import os, shutil, zipfile
from pathlib import Path
from tqdm.notebook import tqdm
import gdown

def count_videos(d):
    return sum(1 for f in Path(d).rglob('*') if f.suffix.lower() in {'.mp4','.avi','.mov','.mkv','.webm'})

def count_images(d):
    return sum(1 for f in Path(d).rglob('*') if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'})

print("Ready.")

In [ ]:
#@title 3. Set Up Kaggle API
#@markdown Get your free API key: https://www.kaggle.com/settings (API → Create New Token)

from google.colab import files as colab_files

KAGGLE_USERNAME = "" #@param {type:"string"}
KAGGLE_KEY = "" #@param {type:"string"}

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
    os.environ['KAGGLE_KEY'] = KAGGLE_KEY
    print(f"Kaggle credentials set for: {KAGGLE_USERNAME}")
else:
    print("Upload your kaggle.json (or paste username/key above):")
    uploaded = colab_files.upload()
    if 'kaggle.json' in uploaded:
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
        print("Kaggle API configured.")
    else:
        print("ERROR: kaggle.json not found. Get it from https://www.kaggle.com/settings")

In [ ]:
#@title 4. Download FaceForensics++ from Kaggle
FFPP_DIR = os.path.join(DRIVE_BASE, "faceforensics++")

print("="*50)
print("FACEFORENSICS++ (Kaggle)")
print("="*50)

if os.path.exists(FFPP_DIR) and count_videos(FFPP_DIR) > 0:
    print(f"Already downloaded ({count_videos(FFPP_DIR)} videos). Skipping.")
else:
    print("Downloading from Kaggle: hungle3401/faceforensics")
    print("This may take 5-15 minutes...\n")

    temp_dir = os.path.join(DRIVE_BASE, "_kaggle_temp")
    os.makedirs(temp_dir, exist_ok=True)

    !kaggle datasets download -d hungle3401/faceforensics -p "$temp_dir" --unzip

    os.makedirs(FFPP_DIR, exist_ok=True)
    for item in Path(temp_dir).iterdir():
        if item.is_dir() and item.name != '__MACOSX':
            dest = os.path.join(FFPP_DIR, item.name)
            if not os.path.exists(dest):
                shutil.move(str(item), dest)
                print(f"  Moved: {item.name}")

    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    print(f"\nDone: {FFPP_DIR}")
    print(f"  Videos: {count_videos(FFPP_DIR)}")
    print(f"  Images: {count_images(FFPP_DIR)}")

In [ ]:
#@title 5. Download Celeb-DF v2 (Google Drive — no approval needed)
#@markdown All files stay on Google Drive. Nothing fills up the VM disk.

CELEBDF_DIR = os.path.join(DRIVE_BASE, "celeb-df")
CELEBDF_ZIP = os.path.join(DRIVE_BASE, "celebdf_v2.zip")

#@markdown Download Celeb-DF v1 as well?
DOWNLOAD_V1 = False #@param {type:"boolean"}

# Direct links from the Celeb-DF authors
CELEBDF_V2_URL = "https://drive.google.com/open?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj"
CELEBDF_V1_URL = "https://drive.google.com/open?id=10NGF38RgF8FZneKOuCOdRIsPzpC7_WDd"

print("="*50)
print("CELEB-DF")
print("="*50)

# --- Celeb-DF v2 ---
print("\nCeleb-DF v2:")

if os.path.exists(CELEBDF_DIR) and count_videos(CELEBDF_DIR) > 0:
    # Already fully extracted
    print(f"  Already extracted ({count_videos(CELEBDF_DIR)} videos). Skipping.")

elif os.path.exists(CELEBDF_ZIP):
    # Zip exists on Drive, just extract it
    print(f"  Zip found on Drive: {os.path.getsize(CELEBDF_ZIP)/(1024**3):.1f} GB")
    print("  Extracting... (this takes a few minutes)")
    os.makedirs(CELEBDF_DIR, exist_ok=True)
    with zipfile.ZipFile(CELEBDF_ZIP, 'r') as zf:
        zf.extractall(CELEBDF_DIR)
    print(f"  Done: {count_videos(CELEBDF_DIR)} videos, {count_images(CELEBDF_DIR)} images")

else:
    # Download zip from Google Drive, then extract
    print("  Downloading from Google Drive (~10 GB)...")
    print("  This may take 3-5 minutes...\n")
    gdown.download(CELEBDF_V2_URL, CELEBDF_ZIP, quiet=False, fuzzy=True)

    print("\n  Extracting...")
    os.makedirs(CELEBDF_DIR, exist_ok=True)
    with zipfile.ZipFile(CELEBDF_ZIP, 'r') as zf:
        zf.extractall(CELEBDF_DIR)
    print(f"  Done: {count_videos(CELEBDF_DIR)} videos, {count_images(CELEBDF_DIR)} images")

# --- Celeb-DF v1 (optional) ---
if DOWNLOAD_V1:
    CELEBDF_V1_DIR = os.path.join(DRIVE_BASE, "celeb-df-v1")
    CELEBDF_V1_ZIP = os.path.join(DRIVE_BASE, "celebdf_v1.zip")
    print("\nCeleb-DF v1:")

    if os.path.exists(CELEBDF_V1_DIR) and count_videos(CELEBDF_V1_DIR) > 0:
        print(f"  Already extracted. Skipping.")
    elif os.path.exists(CELEBDF_V1_ZIP):
        print(f"  Zip found on Drive. Extracting...")
        os.makedirs(CELEBDF_V1_DIR, exist_ok=True)
        with zipfile.ZipFile(CELEBDF_V1_ZIP, 'r') as zf:
            zf.extractall(CELEBDF_V1_DIR)
        print(f"  Done: {count_videos(CELEBDF_V1_DIR)} videos")
    else:
        print("  Downloading from Google Drive...")
        gdown.download(CELEBDF_V1_URL, CELEBDF_V1_ZIP, quiet=False, fuzzy=True)
        print("  Extracting...")
        os.makedirs(CELEBDF_V1_DIR, exist_ok=True)
        with zipfile.ZipFile(CELEBDF_V1_ZIP, 'r') as zf:
            zf.extractall(CELEBDF_V1_DIR)
        print(f"  Done: {count_videos(CELEBDF_V1_DIR)} videos")

In [ ]:
#@title 6. Verify Downloads
print("="*50)
print("DATASET SUMMARY")
print("="*50)

datasets = {
    "FaceForensics++": FFPP_DIR,
    "Celeb-DF v2": CELEBDF_DIR,
}
if DOWNLOAD_V1:
    datasets["Celeb-DF v1"] = os.path.join(DRIVE_BASE, "celeb-df-v1")

for name, path in datasets.items():
    print(f"\n{name}:")
    print(f"  Path: {path}")
    if os.path.exists(path):
        vids = count_videos(path)
        imgs = count_images(path)
        print(f"  Videos: {vids}")
        print(f"  Images: {imgs}")
        for item in sorted(Path(path).iterdir()):
            if item.is_dir():
                sv = count_videos(item)
                si = count_images(item)
                print(f"    {item.name}/ ({sv} videos, {si} images)")
    else:
        print("  NOT DOWNLOADED")

# Show zip files
print("\nZip files on Drive:")
for f in Path(DRIVE_BASE).glob('*.zip'):
    size_gb = f.stat().st_size / (1024**3)
    print(f"  {f.name}: {size_gb:.1f} GB")

print(f"\nTotal Drive usage:")
!du -sh "$DRIVE_BASE" 2>/dev/null

In [ ]:
#@title 7. Delete Zip Files (Optional — saves ~20 GB on Drive)
#@markdown The extracted folders have everything you need. Zips are just for transfer.
#@markdown Delete them after you've verified the extracted folders look good.

DELETE_V2_ZIP = True #@param {type:"boolean"}
DELETE_V1_ZIP = False #@param {type:"boolean"}

zips = []
if DELETE_V2_ZIP:
    zips.append(CELEBDF_ZIP)
if DELETE_V1_ZIP and DOWNLOAD_V1:
    zips.append(os.path.join(DRIVE_BASE, "celebdf_v1.zip"))

for zp in zips:
    if os.path.exists(zp):
        size_gb = os.path.getsize(zp) / (1024**3)
        os.remove(zp)
        print(f"Deleted {os.path.basename(zp)} ({size_gb:.1f} GB)")
    else:
        print(f"Not found: {zp}")

if not zips:
    print("Nothing to delete.")

print(f"\nDrive usage now:")
!du -sh "$DRIVE_BASE" 2>/dev/null